# Stage 16 · Train the three personal-scene arms through update 10,000

Attach the private **personal-input.zip** (or its expanded dataset), select **GPU T4 ×2** and **Run All**. All three arms use the registered 240 × 320 RGB images, seed 42, uniform 512-ray batches and the frozen 75k schedule. Dense uses 64 training cameras and both sparse arms use the same eight. The regularised arm uses both terms. The 12 held-out cameras are reserved for scoring.

The notebook checks actual training rays at 500 and 2,000 before advancing healthy arms to 10,000. The probe samples the central rectangle of the photographs. At each stop it checks gradients, opacity, improvement from initialisation and the black and white controls. From 2,000 onward it also checks against a constant RGB fitted to the same training probe. Each arm stops independently if it fails.

Previous Chair T4 runs measured about 5.4–6.5 updates/s. For three healthy arms the staggered schedule needs roughly 16,000 updates on the busiest GPU, or 41–50 minutes of optimisation at those rates. Allow **45–65 minutes** and use the actual personal-scene rates printed at the first saves to revise this estimate.

Download **personal-s42-stage16-step2000.zip** when offered. Send **personal-s42-stage16-review.zip** after the final wave, including if an arm stops. For a fresh session, attach the most recent saved snapshot alongside the private input and Run All. A notebook-level error produces an interrupted snapshot when possible. Keep all capture outputs private.

In [ ]:
from contextlib import ExitStack
from pathlib import Path
from tempfile import TemporaryDirectory
from uuid import uuid4
import base64
import hashlib
import json
import math
import os
import shutil
import subprocess
import sys
import time
import zipfile

import torch
from IPython.display import HTML, display

inputs = Path("/kaggle/input")
outputs = Path("/kaggle/working")
work = outputs / "recovery-stage16-personal"
work.mkdir(parents=True, exist_ok=True)
if "stage16_temp" not in globals():
    stage16_temp = TemporaryDirectory(prefix="nerf-stage16-")
cache = Path(stage16_temp.name)
arms = ("dense-vanilla", "sparse-vanilla", "sparse-regularized")
view_counts = dict(zip(arms, (64, 8, 8)))
data_hash = "d39c6ff67262904660cf715ea3ba6cdbd17ae3a0cf07a1b83a917cbb50dbe8ae"
code_hash = "4b85a7f9d5836739fb562c5fb3ff25fa68b063073bbea539e38ed1130f088eb5"
review_step = 10_000
device_count = torch.cuda.device_count()
if device_count < 2:
    raise RuntimeError("Select the Kaggle T4 ×2 accelerator and restart this notebook.")
visible = os.environ.get("CUDA_VISIBLE_DEVICES")
devices = [x.strip() for x in visible.split(",")] if visible else [str(i) for i in range(device_count)]
devices = devices[:2]
assert len(set(devices)) == 2
print("Workers:", [torch.cuda.get_device_name(i) for i in range(len(devices))])

def read(path):
    return json.loads(path.read_text(encoding="utf-8"))

def sha(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def unpack(path):
    folder = cache / hashlib.sha256(str(path).encode()).hexdigest()
    folder.mkdir(exist_ok=True)
    with zipfile.ZipFile(path) as archive:
        assert len(archive.namelist()) == len(set(archive.namelist()))
        for name in archive.namelist():
            if not (folder / name).resolve().is_relative_to(folder.resolve()):
                raise ValueError(f"Unexpected ZIP member {name}")
            archive.extract(name, folder)
    return folder

markers = sorted(inputs.rglob("RECOVERY_PERSONAL.txt"))
if markers:
    repo = markers[0].parent
else:
    kits = sorted(inputs.rglob("personal-input.zip"))
    if not kits:
        raise FileNotFoundError("Attach the private personal scene dataset.")
    repo = unpack(kits[0])

assert read(repo / "RECOVERY_PERSONAL.txt") == dict(scene="personal", images=76, data_hash=data_hash, implementation_hash=code_hash)
sys.path.insert(0, str(repo))
from recovery.data import freeze, verify_manifest
from recovery.health import dead_field
from recovery.media import digest
from recovery.training import implementation_hash, load_state, settings, write_json

assert implementation_hash() == code_hash
manifest = work / "manifests/personal.json"
manifest.parent.mkdir(exist_ok=True)
scene = repo/"data/personal"
frozen = verify_manifest(repo/"manifests/personal.json", scene)
assert frozen["data_hash"] == data_hash and frozen["format"] == "nerfstudio" and not frozen["white_background"]
splits = frozen["splits"]
camera_ids = splits["sparse"]
assert [len(splits[k]) for k in ("dense","sparse","test")] == [64,8,12]
assert set(camera_ids)<=set(splits["dense"]) and set(splits["dense"]).isdisjoint(splits["test"])
write_json(manifest, frozen)
cfg = settings("main", frozen["camera"])
plan = dict(cfg, steps=75_000, lr=cfg["lr_final"], prefix_config=cfg, prefix_updates=50_000,
            schedule="50k-exponential-then-25k-hold-v1")
assert cfg["downscale"] == 4 and cfg["batch_size"] == 512 and cfg["steps"] == 50000
from sparse_nerf.data import SceneDataset
from recovery.health import fixed_rays, psnr
controls = {}
for split in ("dense","sparse"):
    ds = SceneDataset(manifest, split, cfg["downscale"])
    assert (ds.width,ds.height)==(240,320)
    ids = list(range(len(ds)))[::max(1,len(ds)//8)][:8]
    _, _, rgb, center = fixed_rays(ds, ids, "cpu")
    targets = rgb[center]
    controls[split] = dict(black=psnr(targets.square().mean()), white=psnr((1-targets).square().mean()),
                           constant=psnr((targets-targets.mean(0)).square().mean()))
print("Verified 64 dense, eight sparse, 12 test cameras. Fixed training-center PSNR controls", controls)

In [ ]:
def relative(arm):
    return Path("runs/personal") / arm / "seed-42"

def checked(run, arm):
    meta = read(run / "run.json")
    assert (meta["scene"], meta["protocol"], meta["seed"], meta["train_views"], meta["data_hash"]) == (
        "personal", arm, 42, view_counts[arm], data_hash)
    assert meta["training_config"] == cfg and meta["downscale"] == 4 and meta["device_type"] == "cuda"
    state = load_state(run, meta)
    step = state["step"]
    assert 0 <= step <= review_step
    slots = state["optimizer"]["state"]
    assert len(slots) == (48 if step else 0) and all(s["step"].item() == step for s in slots.values())
    if step:
        lr = cfg["lr"] * (cfg["lr_final"] / cfg["lr"]) ** ((step-1)/(cfg["steps"]-1))
        assert all(math.isclose(g["lr"], lr, rel_tol=1e-10) for g in state["optimizer"]["param_groups"])
    tensors = [*state["model"].values(), *(s[k] for s in slots.values() for k in ("exp_avg", "exp_avg_sq"))]
    assert all(t.isfinite().all().item() for t in tensors)
    assert all(isinstance(state[k], torch.Tensor) and state[k].numel() for k in ("cpu_rng", "cuda_rng"))
    return step, meta["run_id"]

archives = [p.parent for p in inputs.rglob("stage16-review.json")]
archives += [unpack(p) for p in sorted(inputs.rglob("personal-s42-stage16-*.zip"))]
group_file = work / "group.json"
group_ids = {read(group_file)["group_id"]} if group_file.exists() else set()
snapshots = []
for folder in archives:
    receipt = read(folder / "stage16-review.json")
    if (receipt["data_hash"], receipt["implementation_hash"], receipt["schedule_hash"]) != (data_hash, code_hash, digest(plan)):
        raise ValueError("The attached personal scene snapshot belongs to a different comparison.")
    for name, expected in receipt["sha256"].items():
        assert not Path(name).is_absolute() and ".." not in Path(name).parts
        p = (folder / name).resolve()
        assert p.is_relative_to(folder.resolve()) and sha(p) == expected
    group_ids.add(receipt["group_id"])
    snapshots.append((folder, receipt))
if len(group_ids) > 1:
    raise ValueError("Different personal scene runs are attached. Keep the intended group in a fresh session.")
group_id = next(iter(group_ids)) if group_ids else uuid4().hex
if not group_file.exists():
    write_json(group_file, {"group_id": group_id})

paths = {}
for arm in arms:
    rel = relative(arm)
    options = []
    for root in (work / "instances").glob("*"):
        run = root / rel
        if (run / "checkpoint.pt").is_file():
            step, rid = checked(run, arm)
            options.append((step, 1, run, sha(run / "checkpoint.pt"), rid))
    for folder, receipt in snapshots:
        record = receipt["arms"].get(arm, {})
        if record.get("checkpoint_sha256"):
            src = folder / rel
            step, rid = checked(src, arm)
            ck = sha(src / "checkpoint.pt")
            assert (step, rid, ck) == (record["step"], record["run_id"], record["checkpoint_sha256"])
            options.append((step, 0, src, ck, rid))
    if options:
        step, local, src, ck, rid = max(options)
        if len({o[4] for o in options}) != 1 or len({o[3] for o in options if o[0] == step}) != 1:
            raise ValueError(f"Conflicting {arm} checkpoints.")
        run = src
        if not local:
            run = work / "instances" / ck / rel
            run.parent.mkdir(parents=True, exist_ok=True)
            with TemporaryDirectory(dir=run.parent) as temp:
                staging = Path(temp) / "restored"
                staging.mkdir()
                receipt = next(r for folder, r in snapshots if folder / rel == src)
                members = [Path(name) for name in receipt["sha256"] if Path(name).parent == rel]
                assert {p.name for p in members} >= {"checkpoint.pt", "run.json"}
                if step > 0:
                    assert {p.name for p in members} >= {"health.json", "initial-health.json", "training.jsonl"}
                for member in members:
                    shutil.copyfile(src / member.name, staging / member.name)
                assert checked(staging, arm) == (step, rid)
                staging.rename(run)
        paths[arm] = run
    else:
        step = 0
        paths[arm] = work / "instances" / uuid4().hex / rel
    print(arm, "restored step" if options else "new initialization at step", step)

In [ ]:
def batch(jobs, stop):
    processes = {}
    errors = {}
    with ExitStack() as stack:
        try:
            for arm, gpu in jobs:
                run = paths[arm]
                env = os.environ.copy()
                env.update(CUDA_VISIBLE_DEVICES=gpu, OMP_NUM_THREADS="1", MKL_NUM_THREADS="1", PYTHONUNBUFFERED="1")
                command = [sys.executable, "-u", "-m", "recovery.training", "--preset", "main", "--protocol", arm,
                           "--seed", "42", "--manifest", str(manifest), "--run", str(run), "--device", "cuda:0",
                           "--stop-at", str(stop[arm] if isinstance(stop,dict) else stop), "--save-every", "500"]
                if (run / "checkpoint.pt").is_file():
                    command.append("--resume")
                run.mkdir(parents=True, exist_ok=True)
                # The frozen trainer requires an empty directory for a new run.
                log_path = run.parent / f"{arm}-console.log"
                if not log_path.exists() and (run / "console.log").is_file():
                    shutil.copyfile(run / "console.log", log_path)
                writer = stack.enter_context(log_path.open("ab"))
                reader = stack.enter_context(log_path.open("r", encoding="utf-8", errors="replace"))
                reader.seek(0, os.SEEK_END)
                proc = subprocess.Popen(command, cwd=repo, env=env, stdout=writer, stderr=subprocess.STDOUT)
                processes[arm] = proc, reader
                print(f"Started {arm} on GPU {gpu} through update {stop[arm] if isinstance(stop,dict) else stop:,}.", flush=True)
            while True:
                done = all(proc.poll() is not None for proc, _ in processes.values())
                for arm, (_, reader) in processes.items():
                    for line in reader:
                        print(f"[{arm}] {line.rstrip()}", flush=True)
                if done:
                    break
                time.sleep(2)
            errors = {arm: proc.returncode for arm, (proc, _) in processes.items() if proc.returncode}
        finally:
            for proc, _ in processes.values():
                if proc.poll() is None:
                    proc.terminate()
            for proc, _ in processes.values():
                proc.wait()
    return errors

def inspect_arm(arm):
    run = paths[arm]
    if not (run / "checkpoint.pt").is_file():
        return {"step": None, "passed": False}
    step, rid = checked(run, arm)
    record = dict(step=step, run_id=rid, checkpoint_sha256=sha(run / "checkpoint.pt"), passed=False)
    if not all((run / name).is_file() for name in ("initial-health.json", "health.json")):
        return record
    health, initial = read(run / "health.json"), read(run / "initial-health.json")
    fine = health["fine"]
    record.update(health_step=health["step"], white_psnr=health["white_region_psnr"],
                  fine_psnr=fine["region_psnr"], center_opacity=fine["region_opacity"],
                  outside_opacity=fine["outside_opacity"])
    finite = all(math.isfinite(level[k]) for level in (health["coarse"], health["fine"])
                 for k in ("region_psnr", "region_mse", "avg_density", "region_opacity", "outside_opacity", "density_grad", "color_grad"))
    ref = controls["dense" if arm=="dense-vanilla" else "sparse"]
    assert health["region"]=="central rectangle" and abs(record["white_psnr"]-ref["white"])<1e-4
    record["controls"] = ref
    record["passed"] = bool(finite and step >= 500 and health["step"] == step and not dead_field(health) and
                            fine["region_mse"] < initial["fine"]["region_mse"] and
                            fine["region_psnr"] > max(ref["white"],ref["black"]) + 3 and (step < 2000 or fine["region_psnr"] > ref["constant"]) and fine["region_opacity"] > .02)
    return record

def snapshot(stop, errors, final):
    records = {}
    for arm in arms:
        try:
            records[arm] = inspect_arm(arm)
        except (AssertionError, OSError, RuntimeError, ValueError, KeyError, EOFError) as exc:
            if not errors:
                raise
            records[arm] = {"step": None, "passed": False, "checkpoint_error": f"{type(exc).__name__}: {exc}"}
    tag = "review" if final else ("mixed" if stop == "mixed" else f"step{stop}")
    target = outputs / f"personal-s42-stage16-{tag}.zip"
    files = {"manifests/personal.json": manifest}
    for arm in arms:
        run = paths[arm]
        for name in ("run.json", "checkpoint.pt", "health.json", "initial-health.json"):
            if (run / name).is_file():
                files[(relative(arm) / name).as_posix()] = run / name
        for p in run.glob("train-*.png"):
            files[(relative(arm) / p.name).as_posix()] = p
        log = run.parent / f"{arm}-console.log"
        if not log.is_file():
            log = run / "console.log"
        if log.is_file():
            files[(relative(arm) / "console.log").as_posix()] = log
    receipt = dict(group_id=group_id, scene="personal", seed=42, data_hash=data_hash, implementation_hash=code_hash,
                   schedule_hash=digest(plan), matched_config=plan, sparse_train_ids=camera_ids, train_view_counts=view_counts,
                   review_step=stop, arms=records, process_errors=errors, devices=[torch.cuda.get_device_name(i) for i in range(len(devices))],
                   torch_version=str(torch.__version__), cuda=torch.version.cuda,
                   sha256={name: sha(p) for name,p in files.items()})
    tmp = target.with_suffix(".tmp")
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED, compresslevel=1) as z:
        for name, p in files.items():
            z.write(p, name)
        for arm, record in records.items():
            log = paths[arm] / "training.jsonl"
            if record["step"] is None or not log.is_file():
                continue
            committed = {}
            for line in log.read_text(encoding="utf-8").splitlines():
                try:
                    row = json.loads(line)
                    if row["step"] <= record["step"]:
                        committed[row["step"]] = row
                except (ValueError, KeyError):
                    continue
            name = (relative(arm) / "training.jsonl").as_posix()
            blob = "".join(json.dumps(committed[k], allow_nan=False) + "\n" for k in sorted(committed)).encode()
            receipt["sha256"][name] = hashlib.sha256(blob).hexdigest()
            z.writestr(name, blob)
            if committed:
                record["last_steps_per_s"] = committed[max(committed)]["steps_per_s"]
        z.writestr("stage16-review.json", json.dumps(receipt, indent=2, allow_nan=False))
    with zipfile.ZipFile(tmp) as z:
        assert z.testzip() is None and set(z.namelist()) == set(receipt["sha256"]) | {"stage16-review.json"}
    os.replace(tmp, target)
    for arm, record in records.items():
        print(arm, record, flush=True)
    print(f"Saved {target.name}  SHA-256 {sha(target)}", flush=True)
    encoded = base64.b64encode(target.read_bytes()).decode("ascii")
    display(HTML(f'<a download="{target.name}" href="data:application/zip;base64,{encoded}">Download {target.name}</a>'))
    return records

def run_personal():
    # Each assignment carries its own absolute checkpoint target. A failed arm
    # is excluded from future waves and from fresh-session retries.
    stopped = {}
    for _, receipt in snapshots:
        stopped.update({a:v for a,v in receipt.get("process_errors",{}).items() if a in arms})
    stop_file = work/"stopped.json"
    if stop_file.is_file():
        stopped.update(read(stop_file))
    dense, sparse, reg = arms
    waves = [
        [(dense,500),(sparse,500)], [(reg,500)],
        [(dense,2000),(sparse,2000)], [(reg,2000)],
        [(dense,6000),(sparse,6000)], [(reg,6000),(dense,10000)],
        [(sparse,10000),(reg,10000)],
    ]
    try:
        for i,wave in enumerate(waves):
            goals = dict(wave)
            pending = []
            for arm,goal in wave:
                if arm in stopped:
                    continue
                info=inspect_arm(arm)
                step=info["step"] or 0
                if step>=500 and not info["passed"]:
                    stopped[arm]="restored health check failed"
                elif step<goal:
                    pending.append(arm)
            if pending:
                errors=batch(list(zip(pending,devices)),goals)
                for arm in pending:
                    if arm in errors:
                        stopped[arm]=f"worker exit {errors[arm]}"
                    else:
                        info=inspect_arm(arm)
                        if info["step"]!=goals[arm] or not info["passed"]:
                            stopped[arm]=f"training-center check failed at {info['step']}"
            write_json(stop_file,stopped)
            if i in (1,3,4,5) and (pending or stopped):
                snapshot([0,500,0,2000,6000,"mixed"][i],stopped,False)
        snapshot(review_step,stopped,True)
    except BaseException as exc:
        snapshot(0,dict(stopped,notebook=type(exc).__name__),False)
        raise
    print("Review archive ready. Stopped arms:", stopped)

run_personal()